In [6]:
# See the effects of choosing different ensemble sizes basd on (a) Loss in CRPS
# (b) ESDOF (spatial degree of freedom) concept from Bretherton

# Adapted from daily code
# 2026-10-09 - adjusted file path for using rasters from server
# 2026-10-12 - changed orientation of code from N determination to error representation 

In [7]:
#(a) crps loss
#
# Quantify how much CRPS is lost to finite ensemble size, in degC, at a chosen
# set of ensemble sizes N (see ENSEMBLE_SIZES below).
#
# Why not N* = A / (tol * CRPS(inf))? That quantity depends only on the ratio
# A / CRPS(inf), which is ~1 for any roughly calibrated ensemble -- so N* ~ 1/tol
# whether the predictive distribution is wide (daily) or narrow (monthly), and it
# cannot show the effect of smoothing. The ABSOLUTE loss A/N (degC) does, because
# A itself shrinks when the predictive distribution narrows:
#
#     E[CRPS_hat(N)] = CRPS(inf) + A/N      (Ferro, Richardson & Weigel 2008)
#     loss(N)        = E[CRPS_hat(N)] - CRPS(inf) = A/N
#
# N larger than the number of members on disk (here m=10, so N=12, 15) cannot be
# subsampled. Their loss is the A/N extrapolation, valid because the fair
# estimator of A is unbiased at any m -- provided additional members would behave
# like further iid draws from the same per-case predictive distribution. Those
# rows are flagged "extrapolated" in the printed table.
#
# Optional subsample check (RUN_SUBSAMPLE_CHECK): for N <= m, randomly draw N of
# the m members per case, compute the plain empirical CRPS, and report
# (mean plain CRPS - CRPS(inf)) next to A/N. In expectation this is an algebraic
# identity for subsets drawn without replacement, so it checks the implementation
# (fair-CRPS / A bookkeeping), NOT whether extra members would behave as iid draws.

import os
import json
import numpy as np
import pandas as pd
import rasterio
from tqdm import tqdm
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray as rxr
from scipy.sparse.linalg import eigsh

TIMESCALE = 'monthly'

# Ensemble sizes at which to report the CRPS loss (degC)
ENSEMBLE_SIZES = {
    'daily':   [50, 80, 100, 120, 150],
    'monthly': [5, 8, 10, 12, 15, 50],
}

RUN_SUBSAMPLE_CHECK = True   # empirical check of A/N for N <= m (see above)
N_SUBSAMPLE_REPS = 50        # random subsets averaged per N. Monthly has only a
                              # few thousand cases and m=10, so use many reps
                              # (cheap) to keep Monte Carlo noise small
RANDOM_STATE = 64

START_YEAR, START_MONTH, START_DAY = 2001, 1, 1
END_YEAR, END_MONTH, END_DAY = 2025, 12, 31
start_year, end_year = START_YEAR, END_YEAR
min_valid_days = 20

STN_PATHS = {
    "tmax": {
        "cleaned": r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Insitu\Cleaned\tmax_cleaned.parquet",
        "train_stn": r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Split_stations\tmax_train_stations.csv",
        "test_stn": r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Split_stations\tmax_test_stations.csv",
        "obs_col": "tmax_average",
    },
    "tmin": {
        "cleaned": r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Insitu\Cleaned\tmin_cleaned.parquet",
        "train_stn": r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Split_stations\tmin_train_stations.csv",
        "test_stn": r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Split_stations\tmin_test_stations.csv",
        "obs_col": "tmin_average",
    },
}

tmax_train_df = STN_PATHS["tmax"]["train_stn"]
tmax_test_df = STN_PATHS["tmax"]["test_stn"]
tmin_train_df = STN_PATHS["tmin"]["train_stn"]
tmin_test_df = STN_PATHS["tmin"]["test_stn"]

OUT_DIR = r"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\EnsembleSize_diagnostics"
os.makedirs(OUT_DIR, exist_ok=True)

In [8]:
# 2 Aggregate insitu monthly with completeness filter

insitu_tmax_raw = pd.read_parquet(STN_PATHS["tmax"]["cleaned"])
insitu_tmin_raw = pd.read_parquet(STN_PATHS["tmin"]["cleaned"])

print("==========Aggregating insitu tmax to monthly totals==============")
insitu_tmax_raw["date"] = pd.to_datetime(insitu_tmax_raw[['year', 'month', 'day']])
insitu_tmax_raw = insitu_tmax_raw.rename(columns = {'lat': 'y', 'long': 'x',
                                                            'max_temp': 'tmax'})
insitu_tmax_raw = insitu_tmax_raw[insitu_tmax_raw["tmax"] > -9999]

insitu_tmax_monthly = (
    insitu_tmax_raw
    .groupby(['index_no', 'y', "x", 'year', 'month'])
    .agg(
        tmax_average = ('tmax', 'mean'),
        valid_days = ('tmax', 'count')
    )
    .reset_index()
)

n_before_tmax = len(insitu_tmax_monthly)
insitu_tmax_monthly = insitu_tmax_monthly[insitu_tmax_monthly["valid_days"] >= min_valid_days].copy()
n_after_tmax = len(insitu_tmax_monthly)
print(f"    Station-months before completeness filter: {n_before_tmax:,}")
print(f"    Station-months after (>= {min_valid_days} valid days): {n_after_tmax:,}")
print(f"    Excluded: {n_before_tmax - n_after_tmax:,} ({100*(n_before_tmax-n_after_tmax)/n_before_tmax:.1f}%)")

# Filter to analysis period
insitu_tmax_monthly = insitu_tmax_monthly[
    (insitu_tmax_monthly["year"]    >=  start_year) &
    (insitu_tmax_monthly["year"]    <=  end_year)
].copy()

insitu_tmax_monthly[["x", "y"]] = insitu_tmax_monthly[["x", "y"]].round(5)

# Train/test split
train_ids_tmax = set(pd.read_csv(tmax_train_df)["index_no"])
test_ids_tmax = set(pd.read_csv(tmax_test_df)["index_no"])

insitu_tmax_monthly_train = insitu_tmax_monthly[insitu_tmax_monthly["index_no"].isin(train_ids_tmax)].copy()
insitu_tmax_monthly_test = insitu_tmax_monthly[insitu_tmax_monthly["index_no"].isin(test_ids_tmax)].copy()

print(f"\n  Train station-months: {len(insitu_tmax_monthly_train):,}")
print(f"  Test  station-months: {len(insitu_tmax_monthly_test):,}")

print("\n==========Aggregating insitu tmin to monthly totals==============")
insitu_tmin_raw["date"] = pd.to_datetime(insitu_tmin_raw[['year', 'month', 'day']])
insitu_tmin_raw = insitu_tmin_raw.rename(columns = {'lat': 'y', 'long': 'x',
                                                            'min_temp': 'tmin'})
insitu_tmin_raw = insitu_tmin_raw[insitu_tmin_raw["tmin"] > -9999]

insitu_tmin_monthly = (
    insitu_tmin_raw
    .groupby(['index_no', 'y', "x", 'year', 'month'])
    .agg(
        tmin_average = ('tmin', 'mean'),
        valid_days = ('tmin', 'count')
    )
    .reset_index()
)

n_before_tmin = len(insitu_tmin_monthly)
insitu_tmin_monthly = insitu_tmin_monthly[insitu_tmin_monthly["valid_days"] >= min_valid_days].copy()
n_after_tmin = len(insitu_tmin_monthly)
print(f"    Station-months before completeness filter: {n_before_tmin:,}")
print(f"    Station-months after (>= {min_valid_days} valid days): {n_after_tmin:,}")
print(f"    Excluded: {n_before_tmin - n_after_tmin:,} ({100*(n_before_tmin-n_after_tmin)/n_before_tmin:.1f}%)")

# Filter to analysis period
insitu_tmin_monthly = insitu_tmin_monthly[
    (insitu_tmin_monthly["year"]    >=  start_year) &
    (insitu_tmin_monthly["year"]    <=  end_year)
].copy()

insitu_tmin_monthly[["x", "y"]] = insitu_tmin_monthly[["x", "y"]].round(5)

# Train/test split
train_ids_tmin = set(pd.read_csv(tmin_train_df)["index_no"])
test_ids_tmin = set(pd.read_csv(tmin_test_df)["index_no"])

insitu_tmin_monthly_train = insitu_tmin_monthly[insitu_tmin_monthly["index_no"].isin(train_ids_tmin)].copy()
insitu_tmin_monthly_test = insitu_tmin_monthly[insitu_tmin_monthly["index_no"].isin(test_ids_tmin)].copy()

print(f"\n  Train station-months: {len(insitu_tmin_monthly_train):,}")
print(f"  Test  station-months: {len(insitu_tmin_monthly_test):,}")

monthly_obs_test = {
    "tmax": insitu_tmax_monthly_test,
    "tmin": insitu_tmin_monthly_test,
}

==========Aggregating insitu tmax to monthly totals==============
    Station-months before completeness filter: 14,894
    Station-months after (>= 20 valid days): 14,804
    Excluded: 90 (0.6%)

  Train station-months: 11,310
  Test  station-months: 3,494

==========Aggregating insitu tmin to monthly totals==============
    Station-months before completeness filter: 15,073
    Station-months after (>= 20 valid days): 14,983
    Excluded: 90 (0.6%)

  Train station-months: 11,560
  Test  station-months: 3,423


In [9]:
# HELPERS

def stochastic_raster_path(t_var, period):
    var_dir = t_var[:1].upper() + t_var[1:]
    if TIMESCALE == 'monthly':
        fname = f"{t_var}_QRFSt_{period.year}{period.month:02d}.tif"
    else:
        raise ValueError(f"File Not Found: {TIMESCALE}")
    return fr"\\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\QRF_Products\{var_dir}\Stochastic\{fname}"

# ---------------------------------------------------------------------------
# Fair CRPS (unbiased for CRPS(inf) at any finite m) and its 1/m penalty
# term -- Ferro, Richardson & Weigel (2008)
# ---------------------------------------------------------------------------
def fair_crps_and_penalty(ensemble, obs):
    """
    Returns (fair_crps, A) for one forecast case.
    fair_crps -- unbiased estimate of CRPS(inf); average over many cases to
                 get a stable CRPS(inf) estimate.
    A -- this case's contribution to the finite-ensemble penalty constant;
         average over many cases to get A, then E[CRPS_hat(N)] = CRPS(inf)
         + A/N follows directly, for ANY N (not only the m you have).
    """
    ensemble = np.asarray(ensemble, dtype=float)
    m = ensemble.size
    term1 = np.mean(np.abs(ensemble - obs))
    diffs = np.abs(ensemble[:, None] - ensemble[None, :])
    A = diffs.sum() / (2 * m * (m - 1))
    fair_crps = term1 - A
    return fair_crps, A


def plain_crps_vectorized(ens, obs):
    """
    Plain (biased-at-finite-N) empirical CRPS for many cases at once.
    ens: (n_cases, N) array, obs: (n_cases,) array -> (n_cases,) CRPS values.
    Pairwise term uses the sorted-sample identity
        sum_{i<j} |x_i - x_j| = sum_k (2k - N - 1) * x_(k)      (k = 1..N)
    which is O(N log N) per case instead of building the N x N matrix.
    CRPS_hat = mean|x - y| - (1 / (2 N^2)) * sum_{i,j} |x_i - x_j|
             = mean|x - y| - (sum_{i<j} |x_i - x_j|) / N^2
    """
    n = ens.shape[1]
    term1 = np.abs(ens - obs[:, None]).mean(axis=1)
    s = np.sort(ens, axis=1)
    w = 2 * np.arange(1, n + 1) - n - 1
    pair_sum = (s * w).sum(axis=1)
    return term1 - pair_sum / n ** 2


# ---------------------------------------------------------------------------
# Build the held-out-station comparison table directly from the stochastic
# rasters -- adapted from stochastic_ensemble_validation.py's Section A
# sampling loop, but keeping the FULL member array per case instead of only
# q5/q25/q50/q75/q95.
# ---------------------------------------------------------------------------
def build_comparison_df(t_var):
    test_stn = pd.read_csv(STN_PATHS[t_var]["test_stn"])
    obs_test_df = monthly_obs_test[t_var]
    obs_col = STN_PATHS[t_var]["obs_col"]

    stn_points = list(zip(test_stn['long'], test_stn['lat']))

    if TIMESCALE == 'daily':
        raise ValueError(f"Invalid TIMESCALE Selected")
    else:  # monthly
        periods = pd.period_range(
            start=f"{START_YEAR}-{START_MONTH:02d}",
            end=f"{END_YEAR}-{END_MONTH:02d}",
            freq="M",
        )

    results = []
    missing_files = 0
    for period in tqdm(periods, desc=f"Sampling {TIMESCALE} stochastic rasters for {t_var}"):
        file_path = stochastic_raster_path(t_var, period)
        if not os.path.exists(file_path):
            missing_files += 1
            continue
        with rasterio.open(file_path) as src:
            if src.crs != "EPSG:4326":
                raise ValueError(
                    f"{file_path} is not EPSG:4326 -- station lon/lat sampling would be wrong"
                )
            sampled_values = list(src.sample(stn_points))
            for i, value in enumerate(sampled_values):
                arr = np.asarray(value, dtype=float)
                results.append({
                    'date': f'{period.year}-{period.month}',
                    'year': period.year,
                    'month': period.month,
                    'index_no': test_stn.iloc[i]['index_no'],
                    'ensemble': arr,
                    'n_members': arr.size,
                    'n_nodata': int(np.isnan(arr).sum()),
                })
    if missing_files:
        print(f"  [{t_var}] {missing_files} raster files not found on disk -- skipped")

    if not results:
        return pd.DataFrame()

    sampled_df = pd.DataFrame(results)
    sampled_df['date'] = pd.to_datetime(sampled_df['date'])

    obs_df = obs_test_df[['index_no', 'year', 'month', obs_col]].rename(
        columns={obs_col: 'observation'}
    )

    comparison_df = sampled_df.merge(
        obs_df, on=['year', 'month', 'index_no'], how='inner'
    )
    return comparison_df

def compute_fair_crps_table(comparison_df, expected_m):
    """Drops cases with any nodata pixel or an unexpected member count
    (e.g. a station falling just outside the raster extent on some days),
    then computes per-case fair CRPS and A on the rest."""
    n_before = len(comparison_df)
    clean = comparison_df[
        (comparison_df['n_nodata'] == 0) & (comparison_df['n_members'] == expected_m)
    ].copy()
    n_dropped = n_before - len(clean)
    if n_dropped:
        print(f"  dropped {n_dropped}/{n_before} cases with nodata or member-count mismatch "
              f"(expected m={expected_m})")

    fair_crps_vals = np.empty(len(clean))
    A_vals = np.empty(len(clean))
    for i, (_, row) in enumerate(clean.iterrows()):
        fair_crps_vals[i], A_vals[i] = fair_crps_and_penalty(row['ensemble'], row['observation'])
    clean = clean.assign(fair_crps=fair_crps_vals, A=A_vals)
    return clean


# ---------------------------------------------------------------------------
# Optional empirical check for N <= m (see header: this checks the
# implementation, it is not independent evidence about extra members)
# ---------------------------------------------------------------------------
def subsample_check(clean_df, sizes, m, CRPS_inf, n_reps=N_SUBSAMPLE_REPS, seed=RANDOM_STATE):
    """Returns {N: empirical loss in degC} for every N <= m in `sizes`.
    Loss = mean plain CRPS of random N-member subsets - CRPS(inf)."""
    ens = np.stack(clean_df['ensemble'].to_numpy())           # (n_cases, m)
    obs = clean_df['observation'].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)

    out = {}
    for N in sizes:
        if N > m:
            continue
        if N == m:
            crps_N = plain_crps_vectorized(ens, obs).mean()   # nothing to subsample
        else:
            crps_N = np.mean([
                plain_crps_vectorized(rng.permuted(ens, axis=1)[:, :N], obs).mean()
                for _ in range(n_reps)
            ])
        out[N] = float(crps_N - CRPS_inf)
    return out


def summarize_and_plot(t_var, clean_df, m):
    CRPS_inf = clean_df['fair_crps'].mean()
    A = clean_df['A'].mean()
    n_cases = len(clean_df)
    sizes = ENSEMBLE_SIZES[TIMESCALE]

    print(f"\n[{t_var}] {n_cases} cross-validation cases, {m} members/case")
    print(f"  CRPS(inf) estimate = {CRPS_inf:.4f} °C")
    print(f"  A (finite-ensemble penalty constant) = {A:.4f} °C")
    print(f"  Implied E[CRPS_hat(m={m})] = {CRPS_inf + A/m:.4f} °C  "
          f"(sanity check -- compare to your current plain empirical CRPS at m)")

    empirical = {}
    if RUN_SUBSAMPLE_CHECK:
        empirical = subsample_check(clean_df, sizes, m, CRPS_inf)

    print("  CRPS loss vs. infinite ensemble, loss(N) = A/N:")
    print(f"    {'N':>5} {'loss (°C)':>10} {'% of CRPS(inf)':>15} {'E[CRPS(N)] °C':>14} "
          f"{'subsample loss':>15}  note")
    for N in sizes:
        loss = A / N
        note = "current m" if N == m else ("extrapolated" if N > m else "")
        emp = f"{empirical[N]:15.4f}" if N in empirical else f"{'--':>15}"
        print(f"    {N:>5d} {loss:10.4f} {100 * loss / CRPS_inf:14.2f}% "
              f"{CRPS_inf + loss:14.4f} {emp}  {note}")

    results = {
        "variable": t_var, "timescale": TIMESCALE, "n_cases": int(n_cases),
        "m_current": int(m), "CRPS_inf_degC": float(CRPS_inf), "A_degC": float(A),
        "loss_by_N": {},
    }
    for N in sizes:
        loss = A / N
        results["loss_by_N"][str(N)] = {
            "loss_degC": float(loss),
            "loss_pct_of_CRPS_inf": float(100 * loss / CRPS_inf),
            "expected_crps_degC": float(CRPS_inf + loss),
            "extrapolated": bool(N > m),
            "subsample_loss_degC": empirical.get(N),
        }

    # ---- plot: E[CRPS_hat(N)] vs N, closed-form, with the chosen N marked ----
    N_lo = max(2, int(min(sizes) * 0.5))
    N_hi = int(max(sizes) * 1.25)
    N_range = np.arange(N_lo, N_hi + 1)

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(N_range, CRPS_inf + A / N_range, lw=1.5, label="CRPS(inf) + A/N")
    ax.axhline(CRPS_inf, color="grey", ls=":", lw=1, label=f"CRPS(inf) = {CRPS_inf:.3f} °C")
    ax.scatter(sizes, [CRPS_inf + A / N for N in sizes], color="C0", zorder=3,
               label="chosen N (annotated: loss, °C)")
    for N in sizes:
        ax.annotate(f"+{A / N:.4f}", (N, CRPS_inf + A / N), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=7)
    if empirical:
        ax.scatter(list(empirical), [CRPS_inf + v for v in empirical.values()],
                   facecolors="none", edgecolors="k", s=60, zorder=4, label="subsample check")
    ax.axvline(m, color="firebrick", lw=1.5, label=f"current m={m}")
    ax.set_xlabel("Ensemble size (N)")
    ax.set_ylabel("E[CRPS_hat(N)] (°C)")
    ax.set_title(f"{t_var} ({TIMESCALE}): expected CRPS vs. ensemble size")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plot_path = os.path.join(OUT_DIR, f"{t_var}_{TIMESCALE}_ensemble_size_crps_loss.png")
    fig.savefig(plot_path, dpi=150)
    plt.close(fig)
    print(f"  saved -> {plot_path}")

    summary_path = os.path.join(OUT_DIR, f"{t_var}_{TIMESCALE}_ensemble_size_loss_summary.json")
    with open(summary_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  summary -> {summary_path}")
    return results

In [10]:
# ---------------------------------------------------------------------------
# Main -- runs both tmax and tmin in one pass, matching
# stochastic_ensemble_validation.py's variable loop.
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    if TIMESCALE not in ENSEMBLE_SIZES:
        raise ValueError(f"No ENSEMBLE_SIZES entry for TIMESCALE={TIMESCALE!r}")

    all_results = {}
    print(f"CRPS LOSS BY ENSEMBLE SIZE ({TIMESCALE}) ::::::::::::::")
    for t_var in ["tmax", "tmin"]:
        comparison_df = build_comparison_df(t_var)
        if comparison_df.empty:
            print(f"[{t_var}] no cases found -- check raster paths / date range / TIMESCALE")
            continue
        expected_m = int(comparison_df['n_members'].mode().iloc[0])
        clean_df = compute_fair_crps_table(comparison_df, expected_m=expected_m)
        all_results[t_var] = summarize_and_plot(t_var, clean_df, expected_m)

CRPS LOSS BY ENSEMBLE SIZE (monthly) ::::::::::::::


Sampling monthly stochastic rasters for tmax: 100%|██████████| 300/300 [01:10<00:00,  4.23it/s]



[tmax] 3494 cross-validation cases, 10 members/case
  CRPS(inf) estimate = 0.8871 °C
  A (finite-ensemble penalty constant) = 0.9499 °C
  Implied E[CRPS_hat(m=10)] = 0.9820 °C  (sanity check -- compare to your current plain empirical CRPS at m)
  CRPS loss vs. infinite ensemble, loss(N) = A/N:
        N  loss (°C)  % of CRPS(inf)  E[CRPS(N)] °C  subsample loss  note
        5     0.1900          21.42%         1.0770          0.1905  
        8     0.1187          13.39%         1.0058          0.1186  
       10     0.0950          10.71%         0.9820          0.0950  current m
       12     0.0792           8.92%         0.9662              --  extrapolated
       15     0.0633           7.14%         0.9504              --  extrapolated
       50     0.0190           2.14%         0.9061              --  extrapolated
  saved -> \\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\EnsembleSize_diagnostics\tmax_monthly_ens

Sampling monthly stochastic rasters for tmin: 100%|██████████| 300/300 [01:48<00:00,  2.76it/s]



[tmin] 3423 cross-validation cases, 10 members/case
  CRPS(inf) estimate = 0.7640 °C
  A (finite-ensemble penalty constant) = 0.8934 °C
  Implied E[CRPS_hat(m=10)] = 0.8533 °C  (sanity check -- compare to your current plain empirical CRPS at m)
  CRPS loss vs. infinite ensemble, loss(N) = A/N:
        N  loss (°C)  % of CRPS(inf)  E[CRPS(N)] °C  subsample loss  note
        5     0.1787          23.39%         0.9427          0.1767  
        8     0.1117          14.62%         0.8757          0.1118  
       10     0.0893          11.69%         0.8533          0.0893  current m
       12     0.0744           9.74%         0.8384              --  extrapolated
       15     0.0596           7.80%         0.8236              --  extrapolated
       50     0.0179           2.34%         0.7819              --  extrapolated
  saved -> \\192.168.1.210\giis_home\Unclassified\Jeevan_Katel\ClimateDataWS\Temperature\Data\Processed\Monthly_downscaling\EnsembleSize_diagnostics\tmin_monthly_ens